In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!python preprocess_and_split\ \(2\).py


[sat_] merged -- 434/59,976 rows (0.72%) have a real sat_precipitation_mm value. The rest are NaN (no satellite data collected for that date yet).
[rad_] merged -- 0/59,976 rows (0.00%) have a real rad_dbz value. Radar has no historical archive (live-only source), so this will be at or near 0% until this script is run again after collecting more live snapshots over time.

Train: 46,018 rows (2015-01-01 to 2023-12-31)
Test:  13,958 rows (2024-01-01 to 2026-09-23)

Train label distribution:
risk_category
No Rain     21486
Light       16670
Moderate     7467
Severe        395
Name: count, dtype: int64

Test label distribution:
risk_category
No Rain     6608
Light       4926
Moderate    2275
Severe       149
Name: count, dtype: int64

[IMPORTANT] Training-set sat_ coverage: 0/46,018 rows (0.00%). Training-set rad_ coverage: 0/46,018 rows (0.00%). If either is at or near 0%, the model cannot learn anything meaningful from that source yet -- this is expected given current data collection st

In [4]:
!python /content/drive/MyDrive/PRAVAH_data/train_baseline_model.py

Feature columns resolved dynamically from available data:
  om_ (8 cols): ['om_rainfall_mm', 'om_river_discharge', 'om_rainfall_mm_3d_sum', 'om_rainfall_mm_7d_sum', 'om_rainfall_mm_15d_sum', 'om_river_discharge_3d_sum', 'om_river_discharge_7d_sum', 'om_river_discharge_15d_sum']
  srtm_ (2 cols): ['srtm_elevation_m', 'srtm_slope_deg']
  sat_ (1 cols): ['sat_precipitation_mm']
  rad_ (2 cols): ['rad_dbz', 'rad_rainfall_rate_mm_hr']

[info] No columns found for: ['aws_', 'nwp_'] -- Phase 1/2 data not yet integrated, training on Open-Meteo + SRTM only, as expected today.

Features used this run: ['om_rainfall_mm', 'om_river_discharge', 'om_rainfall_mm_3d_sum', 'om_rainfall_mm_7d_sum', 'om_rainfall_mm_15d_sum', 'om_river_discharge_3d_sum', 'om_river_discharge_7d_sum', 'om_river_discharge_15d_sum', 'srtm_elevation_m', 'srtm_slope_deg', 'sat_precipitation_mm', 'rad_dbz', 'rad_rainfall_rate_mm_hr', 'historical_flood_count']

=== Default (argmax) evaluation ===

              precision    recal

In [ ]:
%cd "/content/drive/MyDrive/PRAVAH/PRAVAH_data"
!ls -la

/content/drive/MyDrive/PRAVAH/PRAVAH_data
total 361673
drwx------ 2 root root      4096 Sep 23 19:06  .
drwx------ 3 root root      4096 Sep 23 19:44  ..
-rw------- 1 root root      8345 Sep 24 16:10  ecmwf_rain_features.csv
-rw------- 1 root root 328397185 Sep 24 16:11  ecmwf_surface.grib2
-rw------- 1 root root  26338794 Sep 24 16:09  ecmwf_surface_pl.grib2
-rw------- 1 root root      1452 Sep 24 08:51  evaluation_report.txt
-rw------- 1 root root      9975 Sep 24 16:00  fetch_ecmwf_rain_data.py
-rw------- 1 root root      6214 Sep 25 10:05  fetch_satellite_data.py
-rw------- 1 root root      5505 Sep 23 19:35 'fetch_terrain_features (1).py'
-rw------- 1 root root      6691 Sep 23 19:34  fetch_training_data.py
-rw------- 1 root root       326 Sep 23 19:38  kerala_terrain_features.csv
-rw------- 1 root root   1900005 Sep 24 08:51 'kerala_test (1).csv'
-rw------- 1 root root   6791114 Sep 24 08:52 'kerala_train (1).csv'
-rw------- 1 root root   6864019 Sep 23 18:58  kerala_training_dat

In [ ]:
!python fetch_training_data.py

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose fetch_ecmwf_rain_data.py from your computer

Saving fetch_ecmwf_rain_data.py to fetch_ecmwf_rain_data.py


In [ ]:
!pip install -q ecmwf-opendata cfgrib xarray pandas eccodes

In [ ]:
!python fetch_ecmwf_rain_data.py --lat 18.63 --lon 73.80 --out ecmwf_rain_features.csv

To ensure the stability of our systems and to preserve resources for our operational activities (network, compute, etc.), access to the open-data portal is limited to 500 simultaneous connections. This limit helps us guarantee reliable service for our operational users, especially during periods of high demand. For added reliability, the open-data is replicated across AWS, Azure, and Google Cloud. If you experience difficulties accessing the portal directly, you can also retrieve the data from these cloud platforms.
By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.
Extracting nearest grid point and building feature table...

[WARNING] These requested variables did not make it into the output: ['10u', '10v', '2d', '2t']
  Check the failure messages above, or that this ECMWF product actually includes them.
Saved 49 rows to ecmwf_rain_features.csv
             step    

In [ ]:
!python preprocess_and_split\ \(1\).py


Train: 46,018 rows (2015-01-01 to 2023-12-31)
Test:  13,958 rows (2024-01-01 to 2026-09-23)

Train label distribution:
risk_category
No Rain     21486
Light       16670
Moderate     7467
Severe        395
Name: count, dtype: int64

Test label distribution:
risk_category
No Rain     6608
Light       4926
Moderate    2275
Severe       149
Name: count, dtype: int64

Suggested class weights:
  Severe: 29.13
  Moderate: 1.54
  Light: 0.69
  No Rain: 0.54

Saved kerala_train.csv and kerala_test.csv

Feature columns detected (11): ['om_rainfall_mm', 'om_river_discharge', 'om_rainfall_mm_3d_sum', 'om_rainfall_mm_7d_sum', 'om_rainfall_mm_15d_sum', 'om_river_discharge_3d_sum', 'om_river_discharge_7d_sum', 'om_river_discharge_15d_sum', 'srtm_elevation_m', 'srtm_slope_deg', 'historical_flood_count']
(Phase 1/2 columns -- aws_*, nwp_*, sat_*, rad_* -- will appear here automatically once added.)


In [ ]:
!pip install xgboost

In [ ]:
!python train_baseline_model.py

In [ ]:
!python fetch_terrain_features.py

Fetching terrain for Thiruvananthapuram...
  [error] elevation query failed after retry: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)
Fetching terrain for Kollam...
  [error] elevation query failed after retry: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)
Fetching terrain for Pathanamthitta...
  [error] elevation query failed after retry: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)
Fetching terrain for Alappuzha...
  [error] elevation query failed after retry: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)
Fetching terrain for Kottayam...
  [error] elevation query failed after retry: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)
Fetching terrain for Idukki...
  [error] elevation query failed after retry: [SSL: CERTIFICATE_VERIF

In [ ]:
!python fetch_terrain_features\ \(1\).py

Fetching terrain for Thiruvananthapuram...
Fetching terrain for Kollam...
Fetching terrain for Pathanamthitta...
Fetching terrain for Alappuzha...
Fetching terrain for Kottayam...
Fetching terrain for Idukki...
Fetching terrain for Ernakulam...
Fetching terrain for Thrissur...
Fetching terrain for Palakkad...
Fetching terrain for Malappuram...
Fetching terrain for Kozhikode...
Fetching terrain for Wayanad...
Fetching terrain for Kannur...
Fetching terrain for Kasaragod...

Saved kerala_terrain_features.csv
          district  elevation_m  slope_deg
Thiruvananthapuram         40.0      0.235
            Kollam         20.0      0.129
    Pathanamthitta         26.0      0.424
         Alappuzha         11.0      0.106
          Kottayam         30.0      0.178
            Idukki        841.0      1.962
         Ernakulam          6.0      0.037
          Thrissur         18.0      0.117
          Palakkad         80.0      0.143
        Malappuram         67.0      0.318
         Kozhik

In [ ]:
!python preprocess_and_split.py

In [ ]:
!pip install -q earthaccess xarray netCDF4 pandas

In [ ]:
import shutil
shutil.rmtree("/content/imerg_raw_tmp", ignore_errors=True)

In [ ]:
!python fetch_satellite_data.py --start 2024-01-01 --end 2024-01-31 --out kerala_satellite_features.csv

Enter your Earthdata Login username: shivangi_patel
Enter your Earthdata password: 
=== 2024-01 (2024-01-01 to 2024-01-31) ===
  Found 31 granules.
  Extracting per-district values...
  Appended 434 rows to kerala_satellite_features.csv

Done. Final file: kerala_satellite_features.csv
Total rows so far: 434
    district        date  sat_precipitation_mm
0  Alappuzha  2024-01-01              0.295000
1  Alappuzha  2024-01-02              0.620000
2  Alappuzha  2024-01-03              5.730000
3  Alappuzha  2024-01-04              6.989999
4  Alappuzha  2024-01-05              3.905001


In [ ]:
import pandas as pd
df = pd.read_csv("kerala_satellite_features.csv")
df.head()

,district,date,sat_precipitation_mm
0,Alappuzha,2024-01-01,0.295000
1,Alappuzha,2024-01-02,0.620000
2,Alappuzha,2024-01-03,5.730000
3,Alappuzha,2024-01-04,6.989999
4,Alappuzha,2024-01-05,3.905001


In [ ]:
!pip install -q requests Pillow pandas

In [ ]:
!python fetch_radar_data.py

Latest radar frame: 2026-09-25T11:10:00+00:00

Appended 14 rows to kerala_radar_features.csv for frame 2026-09-25T11:10:00+00:00
              district  ... rad_rainfall_rate_mm_hr
0   Thiruvananthapuram  ...                    None
1               Kollam  ...                    None
2       Pathanamthitta  ...                    None
3            Alappuzha  ...                    None
4             Kottayam  ...                    None
5               Idukki  ...                    None
6            Ernakulam  ...                    None
7             Thrissur  ...                    None
8             Palakkad  ...                    None
9           Malappuram  ...                    None
10           Kozhikode  ...                    None
11             Wayanad  ...                    None
12              Kannur  ...                    None
13           Kasaragod  ...                    None

[14 rows x 4 columns]

Attribution reminder: RainViewer requires 'Weather data by Rain Vie

In [ ]:
import pandas as pd
df = pd.read_csv("kerala_radar_features.csv")
df.tail(14)  # last run's 14 districts

,district,timestamp_utc,rad_dbz,rad_rainfall_rate_mm_hr
0,Thiruvananthapuram,2026-09-25T11:10:00+00:00,NaN,NaN
1,Kollam,2026-09-25T11:10:00+00:00,NaN,NaN
2,Pathanamthitta,2026-09-25T11:10:00+00:00,NaN,NaN
3,Alappuzha,2026-09-25T11:10:00+00:00,NaN,NaN
4,Kottayam,2026-09-25T11:10:00+00:00,NaN,NaN
5,Idukki,2026-09-25T11:10:00+00:00,NaN,NaN
6,Ernakulam,2026-09-25T11:10:00+00:00,NaN,NaN
7,Thrissur,2026-09-25T11:10:00+00:00,NaN,NaN
8,Palakkad,2026-09-25T11:10:00+00:00,NaN,NaN
9,Malappuram,2026-09-25T11:10:00+00:00,NaN,NaN


In [ ]:
!python clean_satellite_data.py
!python clean_radar_data.py

Loaded 434 rows from kerala_satellite_features.csv
[info] 0/434 rows have missing sat_precipitation_mm -- left as NaN, not imputed

Saved 434 cleaned rows to kerala_satellite_features_clean.csv
    district       date  sat_precipitation_mm
0  Alappuzha 2024-01-01              0.295000
1  Alappuzha 2024-01-02              0.620000
2  Alappuzha 2024-01-03              5.730000
3  Alappuzha 2024-01-04              6.989999
4  Alappuzha 2024-01-05              3.905001
Loaded 14 rows from kerala_radar_features.csv
[info] 14/14 rows missing rad_dbz, 14/14 missing rad_rainfall_rate_mm_hr -- left as NaN, not imputed

Saved 14 cleaned rows to kerala_radar_features_clean.csv
    district             timestamp_utc  rad_dbz  rad_rainfall_rate_mm_hr
0  Alappuzha 2026-09-25 11:10:00+00:00      NaN                      NaN
1  Ernakulam 2026-09-25 11:10:00+00:00      NaN                      NaN
2     Idukki 2026-09-25 11:10:00+00:00      NaN                      NaN
3     Kannur 2026-09-25 11:10:00+